# Edge Detection — the CondadosAI unit, runnable

One notebook, one section per lesson, all on the same image. Run the setup cell,
then jump to whichever lesson you are reading.

| Section | Lesson |
|---|---|
| 1 | What is an edge? |
| 2 | Gradients: central differences, Prewitt, Sobel |
| 3 | Laplacian, LoG and zero-crossings |
| 4 | Canny: non-maximum suppression and hysteresis |
| 5 | Corners: the structure tensor |
| 6 | Your own photo |


## Setup

Versions are pinned so this notebook keeps producing the numbers the articles
publish. Colab ships OpenCV 4.x by default; the pin below replaces it, and Colab
may ask you to restart the runtime afterwards — do that, then skip straight back
to the cell after this one.


In [ ]:
!pip install -q "opencv-python>=5.0.0,<6.0.0" "numpy>=1.26" "matplotlib>=3.8" "loguru>=0.7" "click>=8.1"
!git clone -q https://github.com/CondadosAI/fundamentals-lab.git 2>/dev/null || echo "already cloned"
!pip install -q -e fundamentals-lab

import sys
sys.path.insert(0, "fundamentals-lab/src")


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
print("opencv", cv2.__version__, "| numpy", np.__version__)

from fundamentals_lab.cli.download import download
from click.testing import CliRunner
print(CliRunner().invoke(download, []).output or "dataset ready")


In [ ]:
from fundamentals_lab.core.dataset import canonical_gray, add_noise

gray = canonical_gray()   # 360x480, float32, intensities in 0..1
plt.figure(figsize=(4, 5)); plt.imshow(gray, cmap="gray"); plt.axis("off")
plt.title("the canonical frame"); plt.show()


## 1. What is an edge?

An edge is a place where intensity changes fast. Take one row of pixels and look
at the row itself, its first derivative, and its second: the edge is a peak in the
first and a **zero-crossing** in the second. Change `row` and watch where they move.


In [ ]:
row = gray.shape[0] // 2      # <- change me

profile = gray[row]
first  = np.gradient(profile)
second = np.gradient(first)

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, (d, label) in zip(axes, [(profile, "intensity"), (first, "1st derivative"), (second, "2nd derivative")], strict=True):
    ax.plot(d, lw=1.2); ax.axhline(0, color="0.7", lw=0.8); ax.set_ylabel(label)
axes[-1].set_xlabel(f"column (row {row})"); plt.tight_layout(); plt.show()


Now add noise. The second derivative is the one that falls apart first — it
changes sign several times where there is only one edge. That is the problem
section 3 has to solve.


In [ ]:
rng = np.random.default_rng(0)
noisy = add_noise(gray, 10.0, rng)      # sigma in 0..255 units; try 2, 5, 20

p = noisy[row]
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(p, lw=1); axes[0].set_ylabel("intensity (noisy)")
axes[1].plot(np.gradient(np.gradient(p)), lw=1); axes[1].axhline(0, color="0.7", lw=0.8)
axes[1].set_ylabel("2nd derivative"); plt.tight_layout(); plt.show()


## 2. Gradients

Three operators, same question. They differ only in how they average across the
rows perpendicular to the derivative — and that averaging is the entire difference
in how they survive noise.

They are compared at a **matched edge-pixel count**, not a matched threshold:
Sobel's kernel sums to 8 where the central difference sums to 1, so the same
number means different things.


In [ ]:
from fundamentals_lab.core.gradients import OPERATORS, gradients, magnitude, top_fraction_mask

sigma_noise = 10.0    # <- change me
img = add_noise(gray, sigma_noise, np.random.default_rng(0))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(img, cmap="gray"); axes[0].set_title(f"input, noise sigma={sigma_noise:g}")
for ax, op in zip(axes[1:], OPERATORS, strict=True):
    gx, gy = gradients(img, op)
    ax.imshow(top_fraction_mask(magnitude(gx, gy), 0.05), cmap="gray")
    ax.set_title(f"{op} — top 5%")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


The published sweep says a box average (Prewitt) holds up slightly better than a
triangular one (Sobel) at every noise level. The reason is arithmetic: for the
same three-row support, Prewitt's normalised weights have sum of squares 1/3 and
Sobel's have 3/8, so Prewitt passes about 11% less noise power. What Sobel's
`[1 2 1]` buys is rotational symmetry, not noise rejection.


In [ ]:
w_prewitt = np.array([1, 1, 1]) / 3
w_sobel   = np.array([1, 2, 1]) / 4
print("sum of squares — prewitt", round(float((w_prewitt**2).sum()), 4),
      "| sobel", round(float((w_sobel**2).sum()), 4))


### 2b. How close is this to OpenCV?

The cells printed in the lesson's *How close is this to OpenCV?* section: Sobel in two lines of NumPy, checked against `cv2.Sobel` on a white disc, where the gradient on the rim has to point into the disc.

In [ ]:
import numpy as np
import cv2

# a white disc on black
img = np.zeros((160, 160), np.float32)
cv2.circle(img, (80, 80), 50, 1, -1)

def sobel_x(a):
    # difference along each row
    d = a[:, 2:] - a[:, :-2]
    # [1 2 1] down the columns
    return d[:-2] + 2*d[1:-1] + d[2:]

def sobel_y(a):
    # the same kernel, turned
    return sobel_x(a.T).T

gx, gy = sobel_x(img), sobel_y(img)
mag = np.hypot(gx, gy)
plt.imshow(mag / mag.max(), cmap="gray")
plt.title("our gradient"); plt.axis("off"); plt.show()
print("ours:", gx.shape)

In [ ]:
ref_x = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
ref_y = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)
print("OpenCV:", ref_x.shape)

# compare inside the 1-pixel border
ix = ref_x[1:-1, 1:-1]
iy = ref_y[1:-1, 1:-1]
print("max diff Gx:", np.abs(gx - ix).max())
print("max diff Gy:", np.abs(gy - iy).max())

rim = {"right": (80, 130),
       "left": (80, 30),
       "top": (30, 80),
       "bottom": (130, 80)}
for name, (r, c) in rim.items():
    x, y = ref_x[r, c], ref_y[r, c]
    a = np.degrees(np.arctan2(y, x))
    print(f"{name:6} Gx={x:+.0f} "
          f"Gy={y:+.0f} angle={a:+.0f}")

### 2c. Exercises

**1. Match OpenCV everywhere, on a ramp.** Make `gx_full`, the same size as the image, so that it equals `cv2.Sobel` on every pixel including the border. `np.pad` adds the missing ring; one of its modes copies OpenCV's default border. The last line checks your answer.

In [ ]:
# brightness rises 1 per column
ramp = np.tile(np.arange(64.0), (64, 1))
ramp = ramp.astype(np.float32)

# 62x62 so far: pad ramp first
gx_full = sobel_x(ramp)
print(gx_full.shape)

# checks your answer
ok = gx_full.shape == ramp.shape and np.abs(gx_full - cv2.Sobel(ramp, cv2.CV_32F, 1, 0, ksize=3)).max() < 1e-4
print("correct" if ok else "not yet")

Why the border matters: on a ramp, whose true gradient is the same everywhere, `cv2.Sobel` reports 0 on the first and last columns, because reflecting a ramp makes a peak.

In [ ]:
ref = cv2.Sobel(ramp, cv2.CV_32F, 1, 0, ksize=3)
print(ref[0, [0, 1, 2, -3, -2, -1]])
for mode in ["reflect", "edge", "symmetric"]:
    g = sobel_x(np.pad(ramp, 1, mode=mode))
    print(f"{mode:9}", np.abs(g - ref).max())

**2. Predict, then run.** Change the 2 in `sobel_x` to a 1 (Prewitt) and predict Gx on the right-hand rim before running.

**3. Test a claim the lesson makes:** Sobel's response varies less as an edge rotates, and Prewitt passes less noise. The cell below works both and gives the numbers under *What you should see*.

In [ ]:
def operator(a, w):
    w = np.asarray(w, np.float32) / sum(w)  # normalised to sum to 1
    d = a[:, 2:] - a[:, :-2]
    return w[0] * d[:-2] + w[1] * d[1:-1] + w[2] * d[2:]

yy, xx = np.mgrid[0:101, 0:101].astype(np.float32)
for name, w in [("sobel", [1, 2, 1]), ("prewitt", [1, 1, 1])]:
    peaks = []
    for deg in [0, 15, 30, 45]:
        t = np.radians(deg)
        s = (xx - 50) * np.cos(t) + (yy - 50) * np.sin(t)
        edge = np.clip(s / 2 + 0.5, 0, 1)  # a ramp 2 px wide
        m = np.hypot(operator(edge, w), operator(edge.T, w).T)
        peaks.append(m[40:60, 40:60].max())
    drop = 100 * (peaks[0] - peaks[-1]) / peaks[0]
    print(f"{name:8}", np.round(peaks, 3), f"drop 0->45 deg: {drop:.1f}%")

rng = np.random.default_rng(0)
for name, w in [("sobel", [1, 2, 1]), ("prewitt", [1, 1, 1])]:
    noise = rng.normal(0, 0.1, (200, 200)).astype(np.float32)
    print(f"{name:8} std of Gx on noise: {operator(noise, w).std():.4f}")

## 3. Laplacian, LoG and zero-crossings

Sweep sigma and watch the crossings go from a noisy mesh to a few clean loops.
The floor is computed once from the clean response and reused, so the clean and
noisy panels are actually comparable.


In [ ]:
from fundamentals_lab.core.laplacian import log_response, zero_crossings, crossing_floor, contour_stats

sigmas = [1.0, 1.4, 2.0, 3.0]     # <- change me
fig, axes = plt.subplots(2, len(sigmas), figsize=(4*len(sigmas), 10))
for j, s in enumerate(sigmas):
    rc = log_response(gray, s); floor = crossing_floor(rc)
    rn = log_response(add_noise(gray, 10.0, np.random.default_rng(0)), s)
    mc, mn = zero_crossings(rc, floor), zero_crossings(rn, floor)
    axes[0, j].imshow(mc, cmap="gray"); axes[0, j].set_title(f"clean, sigma={s}")
    axes[1, j].imshow(mn, cmap="gray")
    axes[1, j].set_title(f"noisy — {mn.sum()/mc.sum():.2f}x the crossings")
    print(f"sigma={s}: {contour_stats(mc)}")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()


## 4. Canny

The two ideas, kept apart. Non-maximum suppression thins the ridge; hysteresis
decides who stays. Move `low` and `high` and watch the last panel: raising `low`
breaks contours into fragments, and that is the failure hysteresis exists to fix.


In [ ]:
from fundamentals_lab.core.canny import canny_from_scratch, opencv_canny

sigma, low, high = 1.4, 50, 150       # <- change me
st = canny_from_scratch(gray, sigma=sigma, low=low, high=high)

panels = [(gray, "input"), (st["magnitude"], "gradient magnitude"),
          (st["thin"], "after NMS"), (st["edges"], "after hysteresis")]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (im, t) in zip(axes, panels, strict=True):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

ours, ref = st["edges"], opencv_canny(gray, sigma, low, high)
agree = (ours & ref).sum() / max((ours | ref).sum(), 1)
print(f"IoU with cv2.Canny: {agree:.3f}  (ours {ours.sum()} px, cv2 {ref.sum()} px)")



## 5. Corners

The same 2×2 matrix underneath Harris, Shi-Tomasi and Förstner. Two large
eigenvalues means the patch is pinned down in both directions — which is what
makes it matchable, and where the SIFT unit picks up.


In [ ]:
from fundamentals_lab.core.corners import preset_eigenvalues, harris_response, shi_tomasi_response
from fundamentals_lab.config import PUBLISHED_EIGENVALUES

for name, got in preset_eigenvalues(gray).items():
    pub = PUBLISHED_EIGENVALUES[name]
    print(f"{name:13} l1={got['l1']:.3f} l2={got['l2']:.3f}   published {pub}")

fig, axes = plt.subplots(1, 2, figsize=(10, 7))
for ax, (r, t) in zip(axes, [(harris_response(gray), "Harris"), (shi_tomasi_response(gray), "Shi-Tomasi")], strict=True):
    ax.imshow(r > np.quantile(r, 0.999), cmap="gray"); ax.set_title(f"{t} — top 0.1%"); ax.axis("off")
plt.tight_layout(); plt.show()


## 6. Your own photo

Upload anything and run the whole chain on it. Nothing above depends on the
temple — it is just the image the articles quote numbers from.


In [ ]:
from google.colab import files          # Colab only; skip locally
up = files.upload()
path = next(iter(up))

bgr = cv2.imread(path, cv2.IMREAD_COLOR)
mine = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
st = canny_from_scratch(mine, sigma=1.4, low=50, high=150)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (im, t) in zip(axes, [(mine, "yours"), (st["thin"], "after NMS"), (st["edges"], "edges")], strict=True):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
